# ENSO Magnitude Index

Translates the strength of El Nino episodes (weak to extraordinary) into a 0-1 value per canton and province, using both the maximum and the mean episode strength each unit faced.

Outputs: `enso_magnitude_index_*` tables in `data/processed/`.

## 0. Setup

In [ ]:

# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Tabular data handling
import pandas as pd
# Numerical operations
import numpy as np
# Efficient reader for GeoPackage layers
import pyogrio

# Show up to 40 columns when printing DataFrames
pd.set_option("display.max_columns", 40)

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)

# Create a logger that writes to the logs folder
logger = get_logger("step7_enso_magnitude_index", config, log_dir=PROJECT_ROOT / "logs")
# Log that the notebook started and which config was used
logger.info("magnitud ENSO iniciada — configuración cargada desde %s", CONFIG_PATH)

# ENSO magnitude settings from the config
MAG_CFG = config["enso_magnitude_index"]
# Name of the anchor ENSO index (ICEN)
ANCHOR_COL = MAG_CFG["anchor_enso_index"]
# Mapping from ENSO category to an ordinal value between 0 and 1
CATEGORY_VALUES = MAG_CFG["category_values"]
# Name of the category column of the anchor index
anchor_category_col = f"{ANCHOR_COL.lower()}_category"
# Log the anchor index and the category values
logger.info("Índice ancla: %s | valores de categoría: %s", ANCHOR_COL, CATEGORY_VALUES)

## 1. Combined ENSO magnitude (ICEN + ONI)

In [ ]:
# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Load the extended monthly ENSO series with the combined magnitude
enso_monthly = pd.read_parquet(processed_dir / "enso_index_monthly_extended.parquet")
# Log the number of rows loaded
logger.info("enso_index_monthly_extended cargado: %d filas", len(enso_monthly))

# Check the combined magnitude column exists
assert "magnitud_enso_combinada" in enso_monthly.columns, "recurrencia debe exportar magnitud_enso_combinada (union ICEN+ONI)"
# Use the combined magnitude as the monthly ENSO magnitude
enso_monthly["magnitud_enso"] = enso_monthly["magnitud_enso_combinada"]
# Count months without a magnitude value
n_sin_valor = enso_monthly["magnitud_enso"].isna().sum()
# Check every month has a magnitude
assert n_sin_valor == 0, "recurrencia no deberia dejar meses sin magnitud combinada asignada"
# Log that all 132 months received a magnitude
logger.info("Magnitud ENSO (union ICEN+ONI, calculada en recurrencia) asignada a los 132 meses; sin categorias sin mapear")

# Show the magnitude range per ENSO type
print(enso_monthly.groupby("tipo_enso")["magnitud_enso"].agg(["min", "max", "count"]).sort_values("max"))

## 2. Load events and official names

In [ ]:
# Load the hazard events table without geometry
he = pyogrio.read_dataframe(processed_dir / "hazard_event_geo.gpkg", read_geometry=False)
# Log the number of events loaded
logger.info("hazard_event_geo cargado: %d filas", len(he))

# Load the official canton and province names
adm2 = pyogrio.read_dataframe(
    processed_dir / "admin_boundaries.gpkg", layer="adm2_canton", read_geometry=False
)[["canton_code", "canton_name_official", "province_name_official"]].drop_duplicates()
# Remember the row count to detect accidental changes
n_antes = len(he)
# Attach official canton and province names to each event
he = he.drop(columns=["province_name_official"]).merge(adm2, on="canton_code", how="left")
# Check the join did not change the number of events
assert len(he) == n_antes
# Check every event received an official name
assert he[["canton_name_official", "province_name_official"]].isna().sum().sum() == 0
# Log that official names were attached by canton code
logger.info("Nombres oficiales incorporados por atributo (canton_code); cobertura 100%%")

# Attach the ENSO magnitude of the month to each event
he = he.merge(enso_monthly[["year", "month", "magnitud_enso"]], on=["year", "month"], how="left")
# Check every event received a magnitude
assert he["magnitud_enso"].isna().sum() == 0, "todo evento 2015-2025 debe tener una magnitud ENSO asignada"

# Columns that identify a canton
CANTON_COLS = ["province_code", "province_name_official", "canton_code", "canton_name_official"]
# Columns that identify a province
PROVINCE_COLS = ["province_code", "province_name_official"]

## 3. Aggregation per canton and province: maximum and mean as separate columns

In [ ]:
# Compute the magnitude index for one administrative level
def ejecutar_nivel_magnitud(df: pd.DataFrame, admin_cols: list[str]) -> pd.DataFrame:
    """Compute the ENSO magnitude index for one administrative level, per hazard and combined.
    
    Args:
        df: Event table with the monthly ENSO magnitude.
        admin_cols: Columns that define the administrative level.
    
    Returns:
        Table with the maximum and mean magnitude per unit and view, including units without events (zeros).
    """
    # Universe of all units that have any historical event
    universo = df[admin_cols].drop_duplicates()

    # Compute the magnitude index for one hazard view
    def _por_vista(sub: pd.DataFrame, label: str) -> pd.DataFrame:
        # Maximum and mean magnitude, and event count, per unit
        agg = sub.groupby(admin_cols).agg(
            enso_magnitude_max=("magnitud_enso", "max"),
            enso_magnitude_mean=("magnitud_enso", "mean"),
            n_eventos=("event_id", "count"),
        ).reset_index()
        # Add units with no events in this view
        agg = universo.merge(agg, on=admin_cols, how="left")
        # Missing maximum magnitude becomes zero
        agg["enso_magnitude_max"] = agg["enso_magnitude_max"].fillna(0.0)
        # Missing mean magnitude becomes zero
        agg["enso_magnitude_mean"] = agg["enso_magnitude_mean"].fillna(0.0)
        # Missing event count becomes zero
        agg["n_eventos"] = agg["n_eventos"].fillna(0).astype(int)
        # Label the view
        agg["hazard_type"] = label
        # Return the table for this view
        return agg

    # One table per hazard type
    partes = [_por_vista(df[df.hazard_type == h], h) for h in sorted(df.hazard_type.unique())]
    # Add the combined view of all hazards
    partes.append(_por_vista(df, "TODOS"))
    # Stack all views
    result = pd.concat(partes, ignore_index=True)
    # Check the maximum magnitude is between 0 and 1
    assert result["enso_magnitude_max"].between(0, 1).all(), "enso_magnitude_max fuera de [0,1]"
    # Check the mean magnitude is between 0 and 1
    assert result["enso_magnitude_mean"].between(0, 1).all(), "enso_magnitude_mean fuera de [0,1]"
    # Check the mean never exceeds the maximum
    assert (result["enso_magnitude_mean"] <= result["enso_magnitude_max"] + 1e-9).all(), \
        "el promedio nunca puede superar el máximo — error de cálculo si ocurre"
    # Return the final table
    return result

In [ ]:
# Log the start of the canton-level calculation
logger.info("Calculando Índice de Magnitud ENSO a nivel cantón...")
# Compute the magnitude index at canton level
mag_canton = ejecutar_nivel_magnitud(he, CANTON_COLS)
# Log the start of the province-level calculation
logger.info("Calculando Índice de Magnitud ENSO a nivel provincia...")
# Compute the magnitude index at province level
mag_provincia = ejecutar_nivel_magnitud(he, PROVINCE_COLS)
# Log the number of rows produced
logger.info("Cantón: %d filas | Provincia: %d filas", len(mag_canton), len(mag_provincia))

## 4. Validation

In [ ]:
# Number of hazard types in the data
n_hazard_types = he.hazard_type.nunique()
# Check every canton appears in every view
assert len(mag_canton) == 221 * (n_hazard_types + 1), "cobertura incompleta a nivel cantón"
# Check every province appears in every view
assert len(mag_provincia) == 24 * (n_hazard_types + 1), "cobertura incompleta a nivel provincia"

# Check the canton maximum never exceeds 1
assert mag_canton["enso_magnitude_max"].max() <= 1.0 + 1e-9, "la magnitud combinada no puede superar el techo de la escala ordinal (1.0)"
# Check the canton mean never exceeds 1
assert mag_canton["enso_magnitude_mean"].max() <= 1.0 + 1e-9, "idem para el promedio"

# Combined hazard view at canton level
vista_todos = mag_canton[mag_canton.hazard_type == "TODOS"]

# Log the coverage check
logger.info(
    "Distintos valores en vista TODOS -- máximo: %d/221 | promedio: %d/221",
    vista_todos["enso_magnitude_max"].nunique(), vista_todos["enso_magnitude_mean"].nunique(),
)
# Print a header for the maximum distribution
print("Distribución enso_magnitude_max (vista TODOS):")
# Print how many cantons have each maximum value
print(vista_todos["enso_magnitude_max"].value_counts().sort_index())
# Print an empty line
print()
# Print a header for the mean distribution
print("Distribución enso_magnitude_mean (vista TODOS) -- resumen estadístico:")
# Print summary statistics of the mean
print(vista_todos["enso_magnitude_mean"].describe())
# Print an empty line
print()

# Ten cantons with the highest mean magnitude
top10 = vista_todos.sort_values("enso_magnitude_mean", ascending=False).head(10)
# Print a header for the ranking
print("Top 10 cantones por enso_magnitude_mean:")
# Print the ranking
print(top10[["province_name_official", "canton_name_official", "enso_magnitude_max",
             "enso_magnitude_mean", "n_eventos"]].to_string(index=False))

## 5. Export

In [ ]:
# Tables to export
tablas = {
    "enso_magnitude_index_canton": mag_canton,
    "enso_magnitude_index_provincia": mag_provincia,
}

# Loop over the tables
for nombre, df in tablas.items():
    # Save as Parquet
    df.to_parquet(processed_dir / f"{nombre}.parquet", index=False)
    # Save as CSV
    df.to_csv(processed_dir / f"{nombre}.csv", index=False)
    # Log each exported table
    logger.info("Exportado: %s (%d filas)", nombre, len(df))

# Print the export folder
print(f"Tablas exportadas a {processed_dir} :")
# Loop over the exported tables
for nombre in tablas:
    # Print each table name
    print(f" - {nombre}")